## Ch9-03 -- Stale detection, at scale

This notebook introduces `check_stale()` applied across a small set of tracked records at once, not just one; after running it you can tell, for each of two real records, whether it is still current against the real, committed model, and watch one of them go stale when that model changes under it.

Chapter 8 notebook 03 already showed `check_stale()` on one record against one lemma. This notebook keeps the chapter's original filename (`03-stale-detection.ipynb`): the content is genuinely about scale -- checking several tracked records against the real, current model in one pass -- but "at scale" over two records is still a small, honestly-scoped demonstration, not a claim that every record this tutorial has ever produced is being tracked (notebook 02's own scope note applies here too). It reuses notebook 02's own two records, `AS-C06` and `AS-C08`, rebuilt fresh below since each notebook in this tutorial loads and runs independently.

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch08-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


A record with an empty identifier fails validation outright, distinct from staleness: it was never a valid record to begin with, regardless of which model it references (the same control Chapter 8 notebook 03 used).

In [2]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

broken = ReviewRecord(
    identifier="",        # intentionally empty
    kind="asserted_solution",
    claim="Some claim",
    model_ref="ToasterDemo",
    content_hash=hash_content(source),
    scope="Some scope",
    criteria="Some criteria",
    rationale="Some rationale",
    counterevidence="Some counterevidence",
    record_kind="worked_example",
)
errors = validate_record(broken)
assert len(errors) > 0, "Expected validation errors for empty identifier"
print(f"Negative control ok: errors={errors}")


Negative control ok: errors=['identifier is empty']


`AS-C06` and `AS-C08`, rebuilt below exactly as notebook 02 built them: `AS-C06`'s `content_hash` against `models/ch06-cumulative.sysml`, the model it was actually written against; `AS-C08`'s against `models/ch08-cumulative.sysml`, the real, current model this notebook also loads above.

In [3]:
ch06_source = Path("../../models/ch06-cumulative.sysml").read_text()

as_c06 = ReviewRecord(
    identifier="AS-C06",
    kind="asserted_solution",
    claim=(
        "ResistanceCoil, an electrically switched resistive element that converts "
        "energy to heat by Joule heating, is selected over a combustion-based "
        "alternative (a gas burner, the tongs-and-blowtorch alternative this "
        "tutorial already contrasts) as the mechanism HeatGenerator commits to."
    ),
    model_ref="ToasterDemo::ResistanceCoil",
    content_hash=hash_content(ch06_source),
    scope="ToasterDemo::HeatGenerator and its realizations",
    criteria=(
        "The chosen mechanism must pair with the discrete timing control "
        "ControlSystem's durationOut already provides, and must expose a rating "
        "heatGenerationReq's power threshold can be checked against once a "
        "concrete part exists."
    ),
    rationale=(
        "An electrically resistive element responds to being switched on and off "
        "directly, the same shape as duration's discrete timing signal, while a "
        "combustion-based burner needs separate ignition and fuel-metering "
        "machinery to respond the same way. This selection is a reasoned "
        "engineering preference argued from mechanism, not a claim that the "
        "model already connects one mechanism and not the other."
    ),
    counterevidence=(
        "This does not rule out a combustion design: a burner controlled by its "
        "own timed valve could equally use a duration-like signal. Joule "
        "heating's own relation is still not modeled, so efficiency and "
        "response-time comparisons remain out of reach either way."
    ),
    residual_uncertainties=(
        "Once a supply and a control policy are modeled together, this "
        "selection could be revisited against a real trade study rather than "
        "a domain premise alone."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(as_c06)
print(f"AS-C06 validation errors: {errors}")


AS-C06 validation errors: []


`AS-C08`, the record Chapter 8's own Z3 proof grounds, rebuilt the same way.

In [4]:
as_c08 = ReviewRecord(
    identifier="AS-C08",
    kind="asserted_solution",
    claim=(
        "deliveredEnergyBoundedBySupply, a hand-restated real-arithmetic lemma of the same "
        "shape as HeatGenerator's efficiencyBounded constraint and deliveredEnergy's own "
        "definition, holds for every value of efficiency in [0,1] and every non-negative "
        "power and duration a companion restatement admits."
    ),
    model_ref="ToasterDemo::deliveredEnergyBoundedBySupply",
    content_hash=hash_content(source),
    scope=(
        "The lemma governs the companion restatement's own heatGenCheck.efficiency, "
        "heatGenCheck.power and heatGenCheckDuration features; it is not a solver-checked "
        "reference to HeatGenerator's own efficiencyBounded or deliveredEnergy (DEFERRED.md "
        "D-030, D-031), only a hand-restated copy of the same shape."
    ),
    criteria=(
        "verify_holds() reports a single satisfied verdict for deliveredEnergyBoundedBySupply, "
        "proved for all values of the unbound heatGenCheck.efficiency, heatGenCheck.power and "
        "heatGenCheckDuration features."
    ),
    evidence_refs=[
        "verify_holds: deliveredEnergyBoundedBySupply satisfied (z3, Chapter 8 notebook 02)."
    ],
    rationale=(
        "verify_holds() runs Z3 over the unbound features of a companion restatement of this "
        "lemma and reports it satisfied: proved for all values the restatement admits, a "
        "materially different kind of evidence from a point evaluation."
    ),
    counterevidence=(
        "This proof is NOT a solver-checked reference to HeatGenerator's own "
        "efficiencyBounded constraint or deliveredEnergy calc: this toolchain's Z3 backend "
        "does not compose two separately declared assert constraints (DEFERRED.md D-030), "
        "and cannot reason through a chained calc invocation (D-031). Confirmed directly: "
        "loosening efficiencyBounded's own bound, or doubling deliveredEnergy's own "
        "definition, in the committed model changes neither verdict at all, because the "
        "lemma restates its own copy of both rather than referencing either."
    ),
    residual_uncertainties=(
        "If HeatGenerator's own efficiencyBounded or deliveredEnergy is ever edited, this "
        "record's content_hash does go stale, which forces a re-review, but nothing "
        "automatically re-checks that the restated copy still matches the edited original. "
        "No physical heat generator has been checked against this property."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(as_c08)
print(f"AS-C08 validation errors: {errors}")


AS-C08 validation errors: []


Checked against `models/ch08-cumulative.sysml`, the real, current model both this chapter and Chapter 8 use, before any edit: `AS-C06` was written against `models/ch06-cumulative.sysml`, two chapters' worth of real model growth ago, so it is already stale, honestly and without any edit needed to make it so; `AS-C08` was written against this exact file, so it is still current.

In [5]:
from toaster.evidence import check_stale

tracked = {"AS-C06": as_c06, "AS-C08": as_c08}

print("Before any edit:")
for identifier, record in tracked.items():
    print(f"  {identifier}: stale={check_stale(record, source)}")

assert check_stale(as_c06, source) is True
assert check_stale(as_c08, source) is False


Before any edit:
  AS-C06: stale=True
  AS-C08: stale=False


Chapter 8's own staleness demonstration loosened `deliveredEnergyBoundedBySupply`'s own bound from `<= 1.0` to `<= 1.2`: the model still parses, but the lemma `AS-C08` cites no longer says what the record claims. The same edit, applied here, is reused rather than invented fresh, since it is a real change to a real construct this chapter's own coverage report already reasons about.

In [6]:
revised_source = source.replace(
    "heatGenCheck.efficiency <= 1.0",
    "heatGenCheck.efficiency <= 1.2",
)
assert revised_source != source, "Expected the replacement to change the source"
revised_model = conn.load_from_content(revised_source, strict=False)
assert revised_model.ok, "Revised model should still parse"

print("After loosening deliveredEnergyBoundedBySupply's own bound (1.0 -> 1.2):")
for identifier, record in tracked.items():
    print(f"  {identifier}: stale={check_stale(record, revised_source)}")

assert check_stale(as_c06, revised_source) is True
assert check_stale(as_c08, revised_source) is True
conn.close()


After loosening deliveredEnergyBoundedBySupply's own bound (1.0 -> 1.2):
  AS-C06: stale=True
  AS-C08: stale=True


Both records now report stale, but for different reasons: `AS-C06` was stale before this edit too, for a reason this edit has nothing to do with (two chapters of unrelated model growth); `AS-C08` newly went stale because of exactly the change its own record cites as a risk. Checking a batch of tracked records in one pass is what makes that distinction visible at all: checking either record alone, at either point, would only have reported "stale" or "current" without saying whether the edit just made was the reason.

Two records, tracked and checked together against the same file before and after one real edit, reported two different histories, exactly matching each one's own actual relationship to what changed.

Try the chapter exercise in `exercises/ch09/exercise.ipynb`: it asks you to produce a coverage table for the bread-handling requirements, using the query-and-join pattern notebook 01 builds.